in questo notebook c'é una classe che modifica il terreno piramide in modo che il bot cammini sun una superficie inclinata uniforme, tenere in considerazione per il futuro ma non usare per adesso 

# Microduck: evaluating a walking policy over a suite of conditions

Evaluates the policy you choose on a **suite of fixed conditions** from Phase 1 of the work plan, with the same number of episodes for each condition (100 by default):

- **flat ground** (reference), with random commands and with a constant forward walk;
- **constant slopes walked downhill and uphill** at several angles, also beyond the training range, on a pyramid like the one of the rough-terrain training but with planar faces, with the heading held;
- **terrain types** from section 4.6 (low stairs, uneven ground, waves, ...);
- **reduced friction**, set on **both the floor and the feet**: MuJoCo uses the larger of the two coefficients, so changing only one of them would have no effect;
- **payloads** attached to the trunk, which **shift the centre of mass** (mass and position are chosen per condition).

Each condition runs with the same seed, so two policies evaluated with the same settings see the same conditions (paired comparison).

**What gets saved**, for each condition, in `MyDrive/microduck_thesis/eval/<suite_name>/<condition>/`:

| File | Content | Why this format |
|---|---|---|
| `summary.json` | configuration, software versions, aggregate metrics (falls per 100 episodes with 95% confidence interval, velocity error, power) and one row per episode with outcome and scenario | small and readable, opens anywhere |
| `steps.npz` | one row per control step: observations, hidden-layer activations, action, body motion, feet, torques, power, "falls within 2 s" label | 100 episodes of 20 s are about 100,000 steps of ~1,200 numbers: as JSON that would be several GB and very slow to read. Compressed NumPy with float16 activations is the format proposed in section 4.9 |
| `episode_video.mp4` | video of one of the evaluated episodes (the first one), until it ends or falls | to check by eye what the numbers describe |

Plus `suite_summary.json` in the suite folder: one row per condition, used for the results table and the policy comparison.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. Run cells 1-7 once per session, then cell 8 to run the suite.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Installation

In [3]:
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
os.environ["UV_HTTP_TIMEOUT"] = "600"   # avoids timeouts on the large CUDA packages

if not os.path.exists("/content/microduck_rl"):
    !git clone https://github.com/pollen-robotics/microduck_rl /content/microduck_rl
%cd /content/microduck_rl
!uv sync
!uv run python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.version.cuda)"

downloading uv 0.13.1 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
/content/microduck_rl
Resolved 154 packages in 1ms
Checked 142 packages in 1ms
CUDA: True 12.8


## 3. Offscreen rendering (for the videos)

Colab has no display: MuJoCo renders the video frames on the GPU through EGL. This cell registers the NVIDIA EGL driver if it is missing (the same fix used by the official MuJoCo notebooks) and selects EGL for the evaluation script.

In [4]:
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

## 4. Google Drive

Grant access when asked. Results go to `DRIVE_DIR` (cell 7).

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 5. Choose the policy

Set `POLICY` to one of the names in `POLICIES`. A Hugging Face entry is `(repo, file)`; a local or Drive file is just its path. To add a policy, add a line.

If a download fails with 401/404 the repo is private: run `!uv run huggingface-cli login` with a token that has access. To check the file names in a repo: `!uv run python -c "from huggingface_hub import list_repo_files; print(list_repo_files('<repo>'))"`.

In [6]:
POLICIES = {
    "alpha_walking": ("pollen-robotics/microduck-policies", "alpha_walking.onnx"),
    "rough_walk_g": ("RemiFabre/microduck-rough-walk-g", "policy.onnx"),
    # "my_policy": "/content/drive/MyDrive/policies/my_policy.onnx",
}
POLICY = "alpha_walking"

source = POLICIES[POLICY]
if isinstance(source, str):
    POLICY_PATH = source
else:
    repo, filename = source
    !uv run python -c "from huggingface_hub import hf_hub_download; print(hf_hub_download('{repo}', '{filename}', local_dir='policies/{POLICY}'))"
    POLICY_PATH = os.path.abspath(os.path.join("policies", POLICY, filename))
print("policy:", POLICY, "->", POLICY_PATH, "| found:", os.path.exists(POLICY_PATH))

policies/alpha_walking/alpha_walking.onnx
policy: alpha_walking -> /content/microduck_rl/policies/alpha_walking/alpha_walking.onnx | found: True


## 6. The evaluation script

It evaluates **one condition**: builds the Microduck walking environment (`make_microduck_velocity_env_cfg`) with that condition, converts the ONNX policy into a PyTorch `nn.Sequential` and **checks** it against onnxruntime on real observations (section 4.5; it stops if the difference exceeds 1e-4), records activations with a forward hook after each ELU (section 4.10), runs the episodes in parallel and saves the data. Cell 8 calls it once per condition.

**Video:** during the first round, at every step, the state of robot copy 0 is copied from the GPU simulator into a CPU MuJoCo model of the same scene and rendered with a camera that follows the trunk. The video therefore shows exactly one of the evaluated episodes (episode 0), with the same terrain, ramp and payload, and stops when that episode ends. If rendering fails, the evaluation goes on without the video and a warning is written in `summary.json`.

How each part of a condition is implemented:

| Part | Implementation |
|---|---|
| Slope with direction | A pyramid heightfield like the one of the rough-terrain training, at a fixed angle, as one 40 × 40 m pyramid so that a face is about 19 m long, but with **planar faces** (`FlatFacePyramidTerrainCfg`). mjlab's `HfPyramidSlopedTerrainCfg` builds the height as H·tent(x)·tent(y): its creases run along the centre lines, so a robot on the centre line of a "face" stands on a crease, between two surfaces tilted sideways in opposite directions by about half the nominal slope (in the middle of the face; checked in MuJoCo: ±2.3° at 5°, ±5.1° at 10°, ±7.9° at 15°). Here the height is H·min(tent(x), tent(y)): four planar faces with the creases on the diagonals. The robot is placed on the centre line of the +x face with no random offset, i.e. in the middle of the face both along and across it, where the slope is exactly the nominal one and there is no cross slope. **Uphill and downhill are the same task in opposite directions:** both start in the middle of the face (10 m from the centre, at the computed surface height), with about 9 m available each way; downhill faces the edge of the pyramid, uphill faces the top. Constant forward command with heading hold |
| Heading hold | mjlab's heading controller: every step the yaw-rate command is `clip(heading error, ±1 rad/s)`, with the spawn yaw as target, like an operator correcting with a joystick. Without it the policies drift (`rough_walk_g` up to ~0.3 rad/s, always to the left) and stop walking straight up or down the slope. It changes the command the policy receives, so it must be stated when reporting |
| Ramp (not used) | `FlatRampTerrainCfg` (one long tilted box, from the RollerSlope task) made both policies fall 75-85% of the time already at 2.5°, against 0% on a 5° pyramid face: an artefact of that terrain, so it is not used |
| Terrain type | mjlab terrain generator, 4 × 4 patches of 8 × 8 m, all at the chosen difficulty |
| Friction | the friction of every terrain geom **and** of the two foot geoms is set to the same value; the training foot-friction randomization is switched off for that condition |
| Payload | a rigid body welded to the trunk, with the given mass, placed at the given offset from the trunk centre of mass (x forward, y left, z up): MuJoCo recomputes total mass, centre of mass and inertia |
| Everything else | as in training (centre-of-mass, mass, encoder and IMU randomization); pushes off unless `PUSHES = True` |

The actual foot friction, terrain friction and trunk mass of every episode are read back from the simulator and written in `summary.json`, so you can check that the condition was applied.

In [7]:
%%writefile eval_scenario.py
"""Evaluate a walking policy under one fixed environment condition (work plan, sections 4.6-4.8).

A condition combines a terrain (optionally a slope walked in a fixed direction),
a floor/foot friction, a payload attached to the trunk and a velocity command.
The script runs N episodes of fixed length and records at every control step:
observations, hidden-layer activations, action, body motion, feet, motor
torques, scenario and outcome (fall within 2 s). It also saves a video of the
first evaluated episode.

Usage:
    uv run --with "imageio[ffmpeg]" python eval_scenario.py eval_config.json
    uv run python eval_scenario.py --list        # available terrain types
"""
import dataclasses
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import uuid
from importlib.metadata import version

import mujoco
import numpy as np
import onnx
import onnxruntime as ort
import torch
from onnx import numpy_helper

import mjlab.terrains as terrain_gen
import mjlab_microduck.tasks  # noqa: F401  registers the tasks
from mjlab.envs import ManagerBasedRlEnv
from mjlab.managers import TerminationTermCfg
from mjlab.managers.scene_entity_config import SceneEntityCfg
from mjlab.terrains import heightfield_terrains as hf_terrains
from mjlab.terrains.terrain_generator import TerrainGeneratorCfg, TerrainGeometry, TerrainOutput
from mjlab_microduck.tasks import mdp as microduck_mdp
from mjlab_microduck.tasks.microduck_velocity_env_cfg import make_microduck_velocity_env_cfg
from mjlab_microduck.tasks.slope_terrain import FlatRampTerrainCfg

# ---------------------------------------------------------------------------
# Terrain types (section 4.6). Parameters are scaled for the Microduck, which
# lifts its feet only 1-2 cm; `difficulty` (0-1) picks a point inside each range.
# For types with no parameters below, the parameter names in mjlab 1.3.0 are
# unknown: mjlab's defaults are used, which target larger robots.
# Directional slopes use "slope" with `slope_deg` and `direction` ("downhill" / "uphill"): a
# pyramid heightfield like the one of the rough-terrain training, but with planar faces
# (FlatFacePyramidTerrainCfg), and the robot walks along the centre line of one face. "ramp" (one long tilted box, from the RollerSlope task) is kept only for reference:
# it made both policies fall 75-85% of the time already at 2.5 deg, against 0% on a 5 deg
# pyramid face, so it is not used for the evaluation.
# ---------------------------------------------------------------------------
HF = {"vertical_scale": 0.001}  # 1 mm quantization steps on heightfield surfaces
PYRAMID_PLATFORM_M = 1.0        # side of the flat top platform of the slope pyramid
SLOPE_TILE_SIZE_M = 40.0        # pyramid size for directional slopes: faces about 19 m long, so that from
                                # the middle of a face there are ~9 m both uphill and downhill

CATALOG = {
    "flat": ("BoxFlatTerrainCfg", {}),
    "ramp": (FlatRampTerrainCfg, {}),
    "slope": ("HfPyramidSlopedTerrainCfg", {"slope_range": (0.0, 0.18), "platform_width": 1.0, **HF}),
    "waves": ("HfWaveTerrainCfg", {"amplitude_range": (0.0, 0.02), "num_waves": 4, **HF}),
    "random_uniform": ("HfRandomUniformTerrainCfg", {"noise_range": (0.0, 0.02), "noise_step": 0.005, **HF}),
    "perlin": ("HfPerlinNoiseTerrainCfg", {**HF}),
    "random_grid": ("BoxRandomGridTerrainCfg", {"grid_width": 0.45, "grid_height_range": (0.0, 0.010),
                                                "platform_width": 1.5}),
    "pyramid_stairs": ("BoxPyramidStairsTerrainCfg", {"step_height_range": (0.0, 0.015), "step_width": 0.15,
                                                      "platform_width": 2.0, "border_width": 1.0}),
    "random_stairs": ("BoxRandomStairsTerrainCfg", {"step_height_range": (0.0, 0.015), "step_width": 0.15}),
    "open_stairs": ("BoxOpenStairsTerrainCfg", {"step_height_range": (0.0, 0.015), "step_width": 0.15}),
    "tilted_grid": ("BoxTiltedGridTerrainCfg", {"grid_width": 0.45}),
    "stepping_stones": ("BoxSteppingStonesTerrainCfg", {}),
    "narrow_beams": ("BoxNarrowBeamsTerrainCfg", {}),
    "nested_rings": ("BoxNestedRingsTerrainCfg", {}),
    "random_spread": ("BoxRandomSpreadTerrainCfg", {}),
    "discrete_obstacles": ("HfDiscreteObstaclesTerrainCfg", {}),
}

# Observation layout taken from the mjlab log; used only if the observation
# manager does not expose the term names.
LAYOUT_ACTOR = {"base_ang_vel": [0, 3], "projected_gravity": [3, 6], "joint_pos": [6, 20], "joint_vel": [20, 34],
                "actions": [34, 48], "command": [48, 51], "head_command": [51, 55], "body_command": [55, 61]}
LAYOUT_CRITIC = {"base_lin_vel": [0, 3], "base_ang_vel": [3, 6], "projected_gravity": [6, 9], "joint_pos": [9, 23],
                 "joint_vel": [23, 37], "actions": [37, 51], "command": [51, 54], "foot_height": [54, 56],
                 "foot_air_time": [56, 58], "foot_contact": [58, 60], "foot_contact_forces": [60, 66],
                 "head_command": [66, 70], "body_command": [70, 76]}

FIELD_DESCRIPTIONS = {
    "episode": "episode id (key into 'episodes' in the JSON)",
    "env": "index of the robot copy in the simulator",
    "step": "control step since the start of the episode (50 Hz)",
    "t_s": "time since the start of the episode [s]",
    "obs_actor": "61 observations fed to the policy (layout in meta.obs_layout_actor)",
    "obs_critic": "76 noise-free privileged observations (layout in meta.obs_layout_critic)",
    "action": "14 motor position targets (policy output)",
    "lin_vel_b": "trunk linear velocity in the robot frame [m/s]",
    "ang_vel_b": "trunk angular velocity in the robot frame [rad/s]",
    "gravity_b": "gravity projected in the robot frame (tilt)",
    "foot_contact": "left/right foot contact with the ground",
    "foot_contact_time": "time each foot has been on the ground [s]",
    "foot_air_time": "time each foot has been in the air [s]",
    "foot_height": "height of each foot above the terrain [m] (from the critic foot_height term)",
    "torques": "torques applied by the 14 motors [Nm]",
    "joint_vel": "velocities of the 14 joints [rad/s]",
    "power": "mechanical motor power, sum of |torque x velocity| [W]",
    "vel_error": "|xy velocity - xy command| [m/s]",
    "falls_within_2s": "True if the episode ends with a fall within the next 2 s",
}


# ---------------------------------------------------------------------------
# Policy: ONNX -> PyTorch nn.Sequential (section 4.5)
# ---------------------------------------------------------------------------
class Elementwise(torch.nn.Module):
    """x op c, for the observation normalization baked into the graph."""

    def __init__(self, op, c):
        super().__init__()
        self.op = op
        self.register_buffer("c", torch.as_tensor(np.asarray(c, dtype=np.float32)))

    def forward(self, x):
        if self.op == "Sub":
            return x - self.c
        if self.op == "Add":
            return x + self.c
        if self.op == "Mul":
            return x * self.c
        return x / self.c

    def extra_repr(self):
        return f"{self.op}, shape={tuple(self.c.shape)}"


class Clip(torch.nn.Module):
    def __init__(self, lo, hi):
        super().__init__()
        self.lo, self.hi = lo, hi

    def forward(self, x):
        return torch.clamp(x, self.lo, self.hi)


def load_policy(path):
    """Rebuild the exported MLP as an nn.Sequential (Linear / ELU / normalization)."""
    g = onnx.load(path).graph
    const = {i.name: numpy_helper.to_array(i).copy() for i in g.initializer}
    folding = {"Add": np.add, "Sub": np.subtract, "Mul": np.multiply, "Div": np.divide}
    mods = []
    for n in g.node:
        at = {a.name: onnx.helper.get_attribute_value(a) for a in n.attribute}
        op, ins = n.op_type, list(n.input)
        if op == "Constant":
            const[n.output[0]] = numpy_helper.to_array(at["value"])
            continue
        if all(i in const for i in ins if i):  # node on constants only: fold it now
            v = [const[i] for i in ins if i]
            if op in folding:
                const[n.output[0]] = folding[op](*v)
            elif op == "Sqrt":
                const[n.output[0]] = np.sqrt(v[0])
            elif op in ("Identity", "Cast"):
                const[n.output[0]] = v[0]
            else:
                raise ValueError(f"unsupported ONNX op on constants: {op}")
            continue
        if op == "Gemm":
            assert at.get("alpha", 1.0) == 1.0 and at.get("beta", 1.0) == 1.0 and not at.get("transA", 0), at
            w = const[ins[1]] if at.get("transB", 0) else const[ins[1]].T
            lin = torch.nn.Linear(w.shape[1], w.shape[0], bias=len(ins) > 2 and ins[2] in const)
            with torch.no_grad():
                lin.weight.copy_(torch.from_numpy(np.ascontiguousarray(w, dtype=np.float32)))
                if lin.bias is not None:
                    lin.bias.copy_(torch.from_numpy(const[ins[2]].astype(np.float32).reshape(-1)))
            mods.append(lin)
        elif op == "MatMul":
            w = const[ins[1]].T
            lin = torch.nn.Linear(w.shape[1], w.shape[0], bias=False)
            with torch.no_grad():
                lin.weight.copy_(torch.from_numpy(np.ascontiguousarray(w, dtype=np.float32)))
            mods.append(lin)
        elif op in ("Add", "Sub", "Mul", "Div"):
            if ins[0] in const:
                raise ValueError(f"{op} with the constant as first operand is not supported")
            c = const[ins[1]]
            if op == "Add" and mods and isinstance(mods[-1], torch.nn.Linear) and mods[-1].bias is None:
                mods[-1].bias = torch.nn.Parameter(torch.from_numpy(c.astype(np.float32).reshape(-1)))
            else:
                mods.append(Elementwise(op, c))
        elif op == "Elu":
            mods.append(torch.nn.ELU(alpha=at.get("alpha", 1.0)))
        elif op == "Clip":
            lo = float(const[ins[1]]) if len(ins) > 1 and ins[1] else at.get("min", -np.inf)
            hi = float(const[ins[2]]) if len(ins) > 2 and ins[2] else at.get("max", np.inf)
            mods.append(Clip(lo, hi))
        elif op in ("Identity", "Cast", "Flatten", "Reshape", "Squeeze", "Unsqueeze"):
            continue
        else:
            raise ValueError(f"unsupported ONNX op: {op}")
    return torch.nn.Sequential(*mods).eval()


def check_policy(policy, path, obs):
    """Feed the same inputs to the ONNX model (onnxruntime) and the PyTorch one: outputs must match."""
    sess = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    x = obs[:64].detach()
    with torch.no_grad():
        y_torch = policy(x).cpu().numpy()
    x_np = x.cpu().numpy().astype(np.float32)
    y_onnx = np.concatenate([sess.run(None, {input_name: x_np[i:i + 1]})[0] for i in range(len(x_np))])
    diff = float(np.abs(y_torch - y_onnx).max())
    print(f"[check] ONNX vs PyTorch on {len(x_np)} real observations: max difference {diff:.2e}")
    if diff > 1e-4:
        raise RuntimeError(f"The PyTorch policy does not match the ONNX one (diff {diff:.2e}): data would be unreliable")
    return diff


# ---------------------------------------------------------------------------
# Condition -> environment configuration
# ---------------------------------------------------------------------------
def to_tuple(v):
    """JSON turns tuples into lists: convert them back for mjlab parameters."""
    return tuple(v) if isinstance(v, list) else v


class FlatFacePyramidTerrainCfg(terrain_gen.HfPyramidSlopedTerrainCfg):
    """Pyramid with four PLANAR faces, used for the directional slopes.

    mjlab's HfPyramidSlopedTerrainCfg builds the height map as H * tent(x) * tent(y): its creases run
    along the axes through the centre, not along the diagonals. The centre line of a "face" is then a
    crease between two curved surfaces, each with a cross slope of slope * (1 - d / half size) at a
    distance d from the centre (about half the nominal slope in the middle of the face, in opposite
    directions on the two sides of the crease, which runs a few cm from the centre line, under the
    robot), and no point of that surface has the nominal slope without a cross slope. Here the height is H * min(tent(x), tent(y)): four planar triangular faces,
    creases on the diagonals, and on the centre line of a face exactly the nominal slope with no cross
    slope. Same code as mjlab 1.3.0 (no border, not inverted) with only that line changed.
    """

    def function(self, difficulty, spec, rng):
        if self.inverted or self.border_width > 0:
            raise ValueError("FlatFacePyramidTerrainCfg supports neither inverted pyramids nor borders")
        body = spec.body("terrain")
        slope = self.slope_range[0] + difficulty * (self.slope_range[1] - self.slope_range[0])
        width_pixels = int(self.size[0] / self.horizontal_scale)
        length_pixels = int(self.size[1] / self.horizontal_scale)
        height_max = int(slope * self.size[0] / 2 / self.vertical_scale)
        center_x, center_y = int(width_pixels / 2), int(length_pixels / 2)
        xx = (center_x - np.abs(center_x - np.arange(width_pixels))) / center_x
        yy = (center_y - np.abs(center_y - np.arange(length_pixels))) / center_y
        hf_raw = height_max * np.minimum(xx.reshape(-1, 1), yy.reshape(1, -1))  # mjlab: xx * yy
        platform_width = int(self.platform_width / self.horizontal_scale / 2)
        z_pf = hf_raw[width_pixels // 2 - platform_width, length_pixels // 2 - platform_width]
        hf_raw = np.clip(hf_raw, 0, z_pf)  # flat top platform
        noise = np.rint(hf_raw).astype(np.int16)

        elevation_range = int(noise.max() - noise.min()) or 1
        max_physical_height = elevation_range * self.vertical_scale
        normalized_elevation = (noise - noise.min()) / elevation_range
        unique_id = uuid.uuid4().hex
        field = spec.add_hfield(
            name=f"hfield_{unique_id}",
            size=[self.size[0] / 2, self.size[1] / 2, max_physical_height,
                  max_physical_height * self.base_thickness_ratio],
            nrow=noise.shape[0], ncol=noise.shape[1],
            userdata=normalized_elevation.flatten().astype(np.float32).tolist(),
        )
        material_name = hf_terrains.color_by_height(spec, noise, unique_id, normalized_elevation)
        hfield_geom = body.add_geom(type=mujoco.mjtGeom.mjGEOM_HFIELD, hfieldname=field.name,
                                    pos=[self.size[0] / 2, self.size[1] / 2, 0.0], material=material_name)
        origin = np.array([self.size[0] / 2, self.size[1] / 2, max_physical_height])  # top platform
        flat_patches = hf_terrains._compute_flat_patches(noise, self.vertical_scale, self.horizontal_scale,
                                                         0.0, self.flat_patch_sampling, rng)
        return TerrainOutput(origin=origin, geometries=[TerrainGeometry(geom=hfield_geom, hfield=field)],
                             flat_patches=flat_patches)


def make_terrain(cond, episode_length_s, warnings):
    """Return (TerrainGeneratorCfg, parameters used)."""
    name = cond.get("terrain", "flat")
    if name not in CATALOG:
        raise SystemExit(f"Unknown terrain '{name}'. Available: {', '.join(CATALOG)}")
    difficulty = float(cond.get("difficulty", 1.0))
    overrides = {k: to_tuple(v) for k, v in (cond.get("terrain_params") or {}).items()}

    if name == "ramp":
        # Flat start -> long constant ramp descending towards +x -> flat runout.
        # The robot spawns halfway along the ramp, with room to walk the whole episode
        # in either direction.
        deg = float(cond["slope_deg"])
        speed = abs(float(cond["command"]["vx"])) if isinstance(cond.get("command"), dict) else 0.4
        half = speed * episode_length_s + 1.0
        ramp_len = 2.0 * half
        params = {"flat_length": 1.0, "ramp_length_range": (ramp_len, ramp_len), "runout_length": 3.0,
                  "spawn_on_ramp": half, "deg_min": deg, "deg_max": deg, **overrides}
        sub = FlatRampTerrainCfg(**params)
        size = (params["flat_length"] + ramp_len + params["runout_length"], max(6.0, ramp_len))
        return TerrainGeneratorCfg(size=size, num_rows=1, num_cols=1, curriculum=False,
                                   sub_terrains={"ramp": sub}, add_lights=False), params

    cls, base = CATALOG[name]
    cls = getattr(terrain_gen, cls)
    if name == "slope" and cond.get("direction"):
        cls = FlatFacePyramidTerrainCfg  # planar faces: no ridge under the robot (see the class)
    fields = {f.name for f in dataclasses.fields(cls)}
    params = dict(base)
    if name == "slope" and "slope_deg" in cond:
        s = math.tan(math.radians(float(cond["slope_deg"])))
        params["slope_range"] = (s, s)
        params["platform_width"] = PYRAMID_PLATFORM_M
    params.update(overrides)
    if "proportion" in fields:
        params["proportion"] = 1.0
    try:
        sub = cls(**params)
    except TypeError as e:
        if overrides:
            raise SystemExit(f"{cls.__name__}: invalid parameters ({e}). Available fields: {sorted(fields)}")
        msg = (f"{cls.__name__}: the Microduck-scaled parameters are not valid in this mjlab version ({e}); "
               f"using mjlab defaults. Available fields: {sorted(fields)}")
        print("[WARNING]", msg)
        warnings.append(msg)
        params = {"proportion": 1.0} if "proportion" in fields else {}
        sub = cls(**params)
    if name == "slope" and cond.get("direction"):
        # Directional slope: one large planar-face pyramid, so that a face is long enough to walk on for the whole
        # episode in either direction (8 m patches give 3.5 m faces, after which the robot climbs the
        # next pyramid). One patch is enough: the robot copies are independent simulations.
        size = float(cond.get("tile_size_m", SLOPE_TILE_SIZE_M))
        tile, rows_cols = (size, size), 1
    else:
        # A 4 x 4 grid of 8 m patches, all at the same difficulty: random terrains get 16 different samples
        tile, rows_cols = (8.0, 8.0), 4
    gen = TerrainGeneratorCfg(size=tile, border_width=20.0, num_rows=rows_cols, num_cols=rows_cols,
                              curriculum=False, difficulty_range=(difficulty, difficulty),
                              sub_terrains={name: sub}, add_lights=False)
    return gen, params


def pyramid_face_height(size, slope, platform_width, d, hs=0.1, vs=0.001):
    """Surface height of FlatFacePyramidTerrainCfg on the centre line of the +x face.

    On a face the height depends only on the distance from the centre (min(tent(x), tent(y)) =
    tent(x)), so this rebuilds one row of the height map with the same pixel rounding and platform
    clipping, and interpolates it where MuJoCo puts the samples (n samples spread over the whole
    size, both ends included). Returns (height at distance d from the centre, height of the spawn
    origin of the patch, i.e. of the top platform), in metres.
    """
    n = int(size / hs)
    centre = int(n / 2)
    h_units = int(slope * size / 2 / vs)
    tent = (centre - np.abs(centre - np.arange(n))) / centre
    top_units = h_units * tent[n // 2 - int(platform_width / hs / 2)]
    profile = np.rint(np.clip(h_units * tent, 0, top_units))
    i = (d / size + 0.5) * (n - 1)
    # a flat field (0 deg) still gets a 1-unit hfield height in mjlab: the origin is 1 mm above it
    return float(np.interp(i, np.arange(n), profile)) * vs, max(float(profile.max()), 1.0) * vs


def spec_geoms(spec):
    if hasattr(spec, "geoms"):
        return list(spec.geoms)
    out, stack = [], [spec.worldbody]
    while stack:
        b = stack.pop()
        out += list(b.geoms)
        stack += list(b.bodies)
    return out


def find_trunk(spec):
    try:
        body = spec.body("robot/trunk_base")
        if body is not None:
            return body
    except Exception:  # noqa: BLE001
        pass
    for b in getattr(spec, "bodies", []):
        if b.name.endswith("trunk_base"):
            return b
    raise RuntimeError("trunk_base body not found in the scene spec")


def set_friction(spec, mu):
    """MuJoCo uses the MAXIMUM of the two friction coefficients in a contact:
    set both the floor and the feet, so the effective friction is exactly mu."""
    n_terrain = 0
    for g in spec.body("terrain").geoms:
        g.friction = [mu, g.friction[1], g.friction[2]]
        n_terrain += 1
    feet = [g for g in spec_geoms(spec) if g.name.endswith(("left_foot_collision", "right_foot_collision"))]
    if len(feet) != 2:
        raise RuntimeError(f"expected 2 foot collision geoms, found {[g.name for g in feet]}")
    for g in feet:
        g.friction = [mu, g.friction[1], g.friction[2]]
    print(f"[friction] mu={mu} on {n_terrain} terrain geoms and feet {[g.name for g in feet]}")


def add_payload(spec, mass, offset):
    """Rigid payload welded to the trunk: a child body with a non-colliding sphere of the given
    mass, placed at `offset` (trunk frame: x forward, y left, z up) from the trunk centre of mass.
    MuJoCo recomputes total mass, centre of mass and inertia."""
    trunk = find_trunk(spec)
    com = np.asarray(trunk.ipos if trunk.ipos is not None else np.zeros(3), dtype=float)
    pos = (com + np.asarray(offset, dtype=float)).tolist()
    body = trunk.add_body(name="payload", pos=pos)
    body.add_geom(type=mujoco.mjtGeom.mjGEOM_SPHERE, size=[0.02, 0.0, 0.0], mass=mass,
                  contype=0, conaffinity=0, rgba=[0.85, 0.2, 0.2, 1.0])
    print(f"[payload] {mass} kg at {pos} in the trunk frame (trunk CoM {com.tolist()})")


def add_floor_texture(spec):
    """Checkerboard on every terrain geom (20 cm squares), to see the direction of motion in the
    videos. Textures and materials are only used by the renderer: contacts and dynamics are unchanged."""
    spec.add_texture(name="eval_checker_tex", type=mujoco.mjtTexture.mjTEXTURE_2D,
                     builtin=mujoco.mjtBuiltin.mjBUILTIN_CHECKER, width=512, height=512,
                     rgb1=[0.82, 0.82, 0.82], rgb2=[0.42, 0.45, 0.50])
    # texuniform: the texture repeats per metre; the builtin checker has 2 x 2 squares,
    # so 2.5 repeats per metre give 20 cm squares
    mat = spec.add_material(name="eval_checker_mat", texrepeat=[2.5, 2.5], texuniform=True, reflectance=0.05)
    try:
        textures = list(mat.textures)
        textures[int(mujoco.mjtTextureRole.mjTEXROLE_RGB)] = "eval_checker_tex"
        mat.textures = textures
    except AttributeError:  # older MuJoCo spec API
        mat.texture = "eval_checker_tex"
    n = 0
    for g in spec.body("terrain").geoms:
        g.material = "eval_checker_mat"
        g.rgba = [1.0, 1.0, 1.0, 1.0]  # a non-default geom colour would hide the material
        n += 1
    print(f"[texture] checkerboard on {n} terrain geoms (visual only)")


def make_env(conf, warnings, device):
    cond = conf["condition"]
    num_envs = min(conf["num_episodes"], conf["max_envs"])
    terrain = cond.get("terrain", "flat")

    if terrain == "plane":
        # Exactly the flat environment alpha_walking was trained in (Mjlab-Velocity-Flat-MicroDuck):
        # infinite plane, default contacts and solver settings. Reference for the generated terrains,
        # which use softened contacts and more solver iterations.
        cfg = make_microduck_velocity_env_cfg(play=True, rough=False)
        terrain_params = {"terrain_type": "plane"}
    else:
        terrain_cfg, terrain_params = make_terrain(cond, conf["episode_length_s"], warnings)
        # rough=True: softened terrain contacts, more contacts and solver iterations
        cfg = make_microduck_velocity_env_cfg(play=True, rough=True)
        terrain_cfg.seed = conf["seed"]
        cfg.scene.terrain.terrain_type = "generator"
        cfg.scene.terrain.terrain_generator = terrain_cfg
        cfg.curriculum.pop("terrain_levels", None)
        if terrain == "flat" or (terrain == "slope" and cond.get("direction")):
            # Walking off the generated grid only leads onto the flat border: on flat ground it is
            # not a reason to end the episode (it cut the faster constant-speed walks short). On the
            # directional slope it would cut only the downhill walks short (they move towards the edge
            # of the single pyramid, the uphill ones away from it), so uphill and downhill would no
            # longer be the same task: beyond the edge there is only the flat border, at height 0.
            cfg.terminations.pop("out_of_terrain_bounds", None)
    cfg.scene.num_envs = num_envs
    cfg.seed = conf["seed"]
    cfg.episode_length_s = conf["episode_length_s"] + 5.0  # the time-out must not fire before the end

    # Heading: a fixed spawn yaw
    pose_range = cfg.events["reset_base"].params["pose_range"]
    spawn_yaw = None
    if cond.get("heading_deg") is not None:
        spawn_yaw = math.radians(float(cond["heading_deg"]))
        pose_range["yaw"] = (spawn_yaw, spawn_yaw)

    if terrain == "slope" and cond.get("direction"):
        # Directional slope on the +x face of the planar-face pyramid, on its centre line (y = 0):
        # in the middle of the face, which there is 2 * d wide, with the nominal slope along x and
        # no cross slope. Uphill and downhill are the same task in opposite
        # directions: same start in the middle of the face, at the surface height; downhill faces
        # the edge of the pyramid (+x), uphill faces the top (-x).
        size = float(cond.get("tile_size_m", SLOPE_TILE_SIZE_M))
        slope = math.tan(math.radians(float(cond["slope_deg"])))
        d = float(cond.get("slope_start_m", size / 4))  # middle of the face: halfway from centre to edge
        if cond["direction"] == "downhill":
            spawn_yaw = 0.0
        elif cond["direction"] == "uphill":
            spawn_yaw = math.pi
        else:
            raise SystemExit(f"direction must be 'downhill' or 'uphill', not {cond['direction']!r}")
        h, z_top = pyramid_face_height(size, slope, PYRAMID_PLATFORM_M, d)
        dz = h - z_top  # spawn heights are relative to the patch origin, i.e. the top platform
        z_lo, z_hi = pose_range["z"]
        pose_range["x"] = (d, d)
        pose_range["y"] = (0.0, 0.0)
        pose_range["z"] = (z_lo + dz, z_hi + dz)
        pose_range["yaw"] = (spawn_yaw, spawn_yaw)
        terrain_params["spawn"] = {"x_offset_m": d, "z_offset_m": dz, "yaw_deg": math.degrees(spawn_yaw),
                                   "top_platform_height_m": z_top}
        print(f"[slope] {cond['slope_deg']} deg {cond['direction']}: spawn {d:.2f} m from the pyramid centre, "
              f"{dz:+.3f} m from the top platform height, yaw {math.degrees(spawn_yaw):.0f} deg")

    if cond.get("terrain") == "ramp":
        # no random x/y spawn offset: on a slope it would put the robot inside or above the ramp
        cfg.events["reset_base"].params["pose_range"]["x"] = (0.0, 0.0)
        cfg.events["reset_base"].params["pose_range"]["y"] = (0.0, 0.0)
        # As in the RollerSlope task: out_of_terrain_bounds does not suit the single long ramp tile
        # (it can end episodes right after the spawn), so it is replaced by "fell into the void":
        # the trunk below the lowest flat part of the ramp means the robot left the solid ground.
        cfg.terminations.pop("out_of_terrain_bounds", None)
        drop = terrain_params["ramp_length_range"][1] * math.tan(math.radians(terrain_params["deg_max"]))
        cfg.terminations["fell_into_void"] = TerminationTermCfg(
            func=microduck_mdp.root_height_below,
            params={"min_height": -drop - 0.5, "asset_cfg": SceneEntityCfg("robot", body_names=("trunk_base",))},
        )

    # Velocity command: "random" = training distribution; {"vx": v} = constant forward walk
    command = cond.get("command", "random")
    if isinstance(command, dict):
        c = cfg.commands["twist"]
        c.ranges.lin_vel_x = (float(command["vx"]), float(command["vx"]))
        c.ranges.lin_vel_y = (0.0, 0.0)
        c.ranges.ang_vel_z = (0.0, 0.0)
        c.rel_standing_envs = 0.0
        c.rel_turn_in_place_envs = 0.0
        # mjlab's "forward-only" envs raise vx to at least 0.3 m/s, and "world-frame" envs rotate
        # the command with the robot's yaw: both would change the constant command
        for field in ("rel_forward_envs", "rel_world_envs"):
            if hasattr(c, field):
                setattr(c, field, 0.0)
        if cond.get("hold_heading"):
            # Heading hold, like an operator with a joystick: mjlab's heading controller sets
            # wz = clip(heading_control_stiffness * heading error, ang_vel_z range) every step,
            # with the spawn yaw as target. Without it the policies drift (up to ~0.3 rad/s).
            if spawn_yaw is None:
                raise SystemExit("hold_heading needs a fixed spawn heading (heading_deg or direction)")
            c.heading_command = True
            c.rel_heading_envs = 1.0
            c.ranges.heading = (spawn_yaw, spawn_yaw)
            c.ranges.ang_vel_z = (-1.0, 1.0)  # limits of the corrective yaw-rate command
        else:
            c.rel_heading_envs = 0.0
        cfg.curriculum.pop("standing_envs", None)  # it would reset rel_standing_envs

    # Pushes: in play mode they come every 0.5-1 s; use the training interval or remove them
    if conf["pushes"]:
        cfg.events["push_robot"].interval_range_s = (3.0, 6.0)
    else:
        cfg.events.pop("push_robot", None)

    # Friction and payload are applied to the scene before compilation
    friction = cond.get("friction")
    payload = cond.get("payload")
    if friction is not None:
        cfg.events.pop("foot_friction", None)  # otherwise the training randomization overwrites the feet
    prev_spec_fn = cfg.scene.spec_fn

    floor_texture = conf.get("floor_texture", True)

    def spec_fn(spec):
        if prev_spec_fn is not None:
            prev_spec_fn(spec)
        if friction is not None:
            set_friction(spec, float(friction))
        if payload is not None:
            add_payload(spec, float(payload["mass_kg"]), payload.get("offset_m", [0.0, 0.0, 0.0]))
        if floor_texture:
            try:
                add_floor_texture(spec)
            except Exception as e:  # noqa: BLE001  the texture is cosmetic: never stop the evaluation
                msg = f"floor texture not added: {e}"
                print("[WARNING]", msg)
                warnings.append(msg)

    cfg.scene.spec_fn = spec_fn
    return ManagerBasedRlEnv(cfg=cfg, device=device), num_envs, terrain_params


def obs_layout(env, group, fallback, warnings):
    try:
        om = env.observation_manager
        names, dims = om.active_terms[group], om.group_obs_term_dim[group]
        out, s = {}, 0
        for n, d in zip(names, dims):
            w = int(np.prod(d))
            out[n] = [s, s + w]
            s += w
        return out
    except Exception as e:  # noqa: BLE001
        warnings.append(f"'{group}' layout not read from the observation manager ({e}): using the logged one")
        return fallback


def ids_by_suffix(mj, obj_type, count, suffix):
    return [i for i in range(count) if (mujoco.mj_id2name(mj, obj_type, i) or "").endswith(suffix)]


def scenario_per_env(env, num_envs, warnings):
    """Foot friction, terrain friction, trunk mass, terrain level and column, per copy."""
    out = {}
    mj = env.sim.mj_model
    try:
        feet = (ids_by_suffix(mj, mujoco.mjtObj.mjOBJ_GEOM, mj.ngeom, "left_foot_collision")
                + ids_by_suffix(mj, mujoco.mjtObj.mjOBJ_GEOM, mj.ngeom, "right_foot_collision"))
        gf = torch.as_tensor(env.sim.model.geom_friction)
        f = gf[:, feet, 0] if gf.ndim == 3 else gf[feet, 0].expand(num_envs, -1)
        out["foot_friction"] = f.cpu().numpy()
        terrain_bodies = ids_by_suffix(mj, mujoco.mjtObj.mjOBJ_BODY, mj.nbody, "terrain")
        terrain_geoms = [i for i in range(mj.ngeom) if mj.geom_bodyid[i] in terrain_bodies]
        out["terrain_friction"] = float(mj.geom_friction[terrain_geoms, 0].max()) if terrain_geoms else None
    except Exception as e:  # noqa: BLE001
        warnings.append(f"friction not read: {e}")
    try:
        trunk = ids_by_suffix(mj, mujoco.mjtObj.mjOBJ_BODY, mj.nbody, "trunk_base")[0]
        bm = torch.as_tensor(env.sim.model.body_mass)
        m = bm[:, trunk] if bm.ndim == 2 else bm[trunk].expand(num_envs)
        out["trunk_mass"] = m.cpu().numpy()
        out["robot_mass_nominal"] = float(mj.body_subtreemass[trunk])  # includes the payload
    except Exception as e:  # noqa: BLE001
        warnings.append(f"mass not read: {e}")
    terrain = getattr(env.scene, "terrain", None)
    for key, attr in (("level", "terrain_levels"), ("column", "terrain_types")):
        v = getattr(terrain, attr, None)
        if v is None:
            warnings.append(f"{attr} not available on the scene terrain")
            out[key] = np.full(num_envs, -1)
        else:
            out[key] = torch.as_tensor(v).cpu().numpy().astype(int)
    return out


def software_versions():
    try:
        commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
    except Exception:  # noqa: BLE001
        commit = None
    versions = {"microduck_rl_commit": commit, "torch": torch.__version__, "mujoco": mujoco.__version__}
    for pkg in ("mjlab", "mujoco-warp", "warp-lang", "onnxruntime"):
        try:
            versions[pkg] = version(pkg)
        except Exception:  # noqa: BLE001
            versions[pkg] = None
    return versions


def stats(x):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return None
    return {"mean": float(x.mean()), "min": float(x.min()), "max": float(x.max()), "std": float(x.std())}


class EpisodeVideo:
    """Video of one evaluated episode (robot copy 0).

    At every control step the state of copy 0 is copied from the GPU simulator into a
    CPU MuJoCo model of the same scene (terrain, ramp and payload included) and rendered
    with a camera that follows the trunk. Rendering needs MUJOCO_GL=egl on Colab.
    """

    def __init__(self, env, width=640, height=480):
        self.model = env.sim.mj_model
        self.data = mujoco.MjData(self.model)
        self.qpos = env.sim.data.qpos
        self.renderer = mujoco.Renderer(self.model, height, width)
        self.camera = mujoco.MjvCamera()
        self.camera.type = mujoco.mjtCamera.mjCAMERA_TRACKING
        self.camera.trackbodyid = ids_by_suffix(self.model, mujoco.mjtObj.mjOBJ_BODY, self.model.nbody,
                                                "trunk_base")[0]
        self.camera.distance = 0.9
        self.camera.elevation = -15.0
        self.camera.azimuth = 120.0  # three-quarter side view: shows the slope and the gait
        self.frames = []

    def capture(self):
        self.data.qpos[:] = self.qpos[0].detach().cpu().numpy()
        mujoco.mj_forward(self.model, self.data)
        self.renderer.update_scene(self.data, camera=self.camera)
        self.frames.append(self.renderer.render().copy())

    def save(self, path, fps):
        import imageio  # installed with: uv run --with "imageio[ffmpeg]" ...
        imageio.mimsave(path, self.frames, fps=fps)
        self.renderer.close()


def wilson_ci(k, n, z=1.96):
    """95% Wilson confidence interval of a proportion, in percent."""
    if n == 0:
        return None
    p = k / n
    den = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / den
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return [100.0 * max(0.0, centre - half), 100.0 * min(1.0, centre + half)]


# ---------------------------------------------------------------------------
# Evaluation
# ---------------------------------------------------------------------------
def main(conf):
    t_start = time.time()
    device = "cuda:0" if torch.cuda.is_available() else "cpu"
    torch.manual_seed(conf["seed"])
    np.random.seed(conf["seed"])
    warnings = []
    cond = conf["condition"]
    print(f"=== condition '{cond['name']}' | policy {os.path.basename(conf['policy'])} | "
          f"{conf['num_episodes']} episodes ===")

    env, num_envs, terrain_params = make_env(conf, warnings, device)
    dt = env.step_dt
    T = int(round(conf["episode_length_s"] / dt))
    steps_2s = int(round(2.0 / dt))
    robot = env.scene["robot"]
    feet_sensor = env.scene.sensors["feet_ground_contact"]
    layout_a = obs_layout(env, "actor", LAYOUT_ACTOR, warnings)
    layout_c = obs_layout(env, "critic", LAYOUT_CRITIC, warnings)
    cmd_slice = slice(*layout_a.get("command", [48, 51]))

    policy = load_policy(conf["policy"]).to(device)
    print(policy)
    elus = [m for m in policy if isinstance(m, torch.nn.ELU)]
    if not elus:
        raise RuntimeError("no ELU hidden layer found in the policy")
    hidden_names = [f"h{k + 1}" for k in range(len(elus))]
    activations = {}
    for k, m in enumerate(elus):  # forward hook (section 4.10): store each ELU output
        m.register_forward_hook(lambda mod, i, o, k=k: activations.__setitem__(k, o.detach()))

    video = None
    if conf.get("save_video", True):
        try:
            video = EpisodeVideo(env)
        except Exception as e:  # noqa: BLE001
            msg = f"video disabled, the renderer could not start ({e}); is MUJOCO_GL=egl set?"
            print("[WARNING]", msg)
            warnings.append(msg)

    save_steps = conf["save_steps"]
    rows = {}

    def append(name, x):
        rows.setdefault(name, []).append(x.detach().to("cpu"))

    episodes, num_done, round_idx, check_diff, scen = [], 0, 0, None, {}
    while num_done < conf["num_episodes"]:
        num_now = min(num_envs, conf["num_episodes"] - num_done)
        obs, _ = env.reset(seed=conf["seed"] + round_idx)
        if check_diff is None:
            check_diff = check_policy(policy, conf["policy"], obs["actor"])
        scen = scenario_per_env(env, num_envs, warnings)
        alive = torch.zeros(num_envs, dtype=torch.bool, device=device)
        alive[:num_now] = True
        ep_id = torch.arange(num_envs, device=device) + num_done
        end_step = [-1] * num_envs
        outcome = ["completed"] * num_envs
        err_sum = torch.zeros(num_envs, device=device)
        power_sum = torch.zeros(num_envs, device=device)
        step_count = torch.zeros(num_envs, device=device)
        max_tilt = torch.zeros(num_envs, device=device)
        # velocity tracking averaged over ~1 s (an exponential moving average), which removes the
        # oscillation of the trunk within each step; counted after a 1 s warm-up
        err_1s_sum = torch.zeros(num_envs, device=device)
        err_1s_count = torch.zeros(num_envs, device=device)
        fwd_vel_sum = torch.zeros(num_envs, device=device)
        cmd_fwd_sum = torch.zeros(num_envs, device=device)
        yaw_rate_sum = torch.zeros(num_envs, device=device)  # integrated: heading drift of the episode
        ema_alpha, warmup_steps = dt / 1.0, int(round(1.0 / dt))

        for t in range(T):
            if not alive.any():
                break
            idx = alive.nonzero(as_tuple=True)[0]
            if video is not None and round_idx == 0 and bool(alive[0]):
                try:
                    video.capture()  # the first episode of copy 0, frame by frame until it ends
                except Exception as e:  # noqa: BLE001
                    warnings.append(f"video stopped at step {t}: {e}")
                    video = None
            a_obs, c_obs = obs["actor"], obs["critic"]
            with torch.no_grad():
                action = policy(a_obs)  # the hooks fill `activations`

            # ground truth at the same instant as the observations (before the step)
            d = robot.data
            tau, qd = d.actuator_force[:, :14], d.joint_vel[:, :14]
            power = (tau * qd).abs().sum(-1)
            vel_xy = d.root_link_lin_vel_b[:, :2]
            cmd_xy = a_obs[:, cmd_slice][:, :2]
            vel_err = (vel_xy - cmd_xy).norm(dim=-1)  # instantaneous: includes the gait oscillation
            zero = torch.zeros_like(vel_err)
            err_sum += torch.where(alive, vel_err, zero)
            power_sum += torch.where(alive, power, zero)
            step_count += alive.float()
            fwd_vel_sum += torch.where(alive, vel_xy[:, 0], zero)
            cmd_fwd_sum += torch.where(alive, cmd_xy[:, 0], zero)
            yaw_rate_sum += torch.where(alive, d.root_link_ang_vel_b[:, 2], zero)

            pos_xy = d.root_link_pos_w[:, :2]
            if t == 0:
                start_pos, last_pos = pos_xy.clone(), pos_xy.clone()
                vel_ema, cmd_ema = vel_xy.clone(), cmd_xy.clone()
            last_pos = torch.where(alive[:, None], pos_xy, last_pos)
            vel_ema += ema_alpha * (vel_xy - vel_ema)
            cmd_ema += ema_alpha * (cmd_xy - cmd_ema)
            if t >= warmup_steps:
                err_1s = (vel_ema - cmd_ema).norm(dim=-1)
                err_1s_sum += torch.where(alive, err_1s, zero)
                err_1s_count += alive.float()
            # trunk tilt from vertical: catches near-falls that the fall termination does not count
            g = d.projected_gravity_b
            tilt = torch.rad2deg(torch.acos((-g[:, 2] / g.norm(dim=-1).clamp_min(1e-6)).clamp(-1.0, 1.0)))
            max_tilt = torch.where(alive, torch.maximum(max_tilt, tilt), max_tilt)

            if save_steps:
                n = len(idx)
                append("episode", ep_id[idx].int())
                append("env", idx.short())
                append("step", torch.full((n,), t, dtype=torch.int16))
                append("obs_actor", a_obs[idx])
                append("obs_critic", c_obs[idx])
                for k, name in enumerate(hidden_names):
                    append(name, activations[k][idx].half())
                append("action", action[idx])
                append("lin_vel_b", d.root_link_lin_vel_b[idx])
                append("ang_vel_b", d.root_link_ang_vel_b[idx])
                append("gravity_b", d.projected_gravity_b[idx])
                for name, attr in (("foot_contact", "found"), ("foot_contact_time", "current_contact_time"),
                                   ("foot_air_time", "current_air_time")):
                    v = getattr(feet_sensor.data, attr, None)
                    if v is not None:
                        append(name, v[idx].float().reshape(n, -1))
                append("torques", tau[idx])
                append("joint_vel", qd[idx])
                append("power", power[idx])
                append("vel_error", vel_err[idx])

            obs, _, term, trunc, _ = env.step(action)
            finished = alive & (term | trunc)
            for i in finished.nonzero(as_tuple=True)[0].tolist():
                end_step[i] = t
                outcome[i] = "fell" if bool(term[i]) else "truncated"  # truncated = time-out or off the terrain
            alive &= ~finished

        displacement = (last_pos - start_pos).norm(dim=-1)
        for i in range(num_now):
            n_i = max(float(step_count[i]), 1.0)
            duration = (end_step[i] + 1 if end_step[i] >= 0 else T) * dt
            episodes.append({
                "id": int(ep_id[i]),
                "env": i,
                "round": round_idx,
                "terrain_level": int(scen["level"][i]),
                "terrain_column": int(scen["column"][i]),
                "outcome": outcome[i],
                "duration_s": duration,
                "fall_time_s": (end_step[i] + 1) * dt if outcome[i] == "fell" else None,
                "mean_vel_error_m_s": float(err_sum[i]) / n_i,
                "mean_vel_error_1s_m_s": (float(err_1s_sum[i]) / float(err_1s_count[i])
                                          if float(err_1s_count[i]) > 0 else None),
                "mean_forward_vel_m_s": float(fwd_vel_sum[i]) / n_i,
                "mean_cmd_forward_vel_m_s": float(cmd_fwd_sum[i]) / n_i,
                "displacement_m": float(displacement[i]),
                "mean_speed_m_s": float(displacement[i]) / duration,
                "mean_yaw_rate_rad_s": float(yaw_rate_sum[i]) / n_i,          # > 0 = turning left
                "heading_change_deg": math.degrees(float(yaw_rate_sum[i]) * dt),
                "mean_power_W": float(power_sum[i]) / n_i,
                "max_tilt_deg": float(max_tilt[i]),
                "foot_friction": scen["foot_friction"][i].tolist() if "foot_friction" in scen else None,
                "terrain_friction": scen.get("terrain_friction"),
                "trunk_mass_kg": float(scen["trunk_mass"][i]) if "trunk_mass" in scen else None,
            })
        num_done += num_now
        round_idx += 1
        falls = sum(e["outcome"] == "fell" for e in episodes)
        print(f"[round {round_idx}] episodes {num_done}/{conf['num_episodes']} | falls so far {falls} | "
              f"{time.time() - t_start:.0f} s")

    # ---------------- summary ----------------
    n_ep = len(episodes)
    fallen = [e for e in episodes if e["outcome"] == "fell"]
    summary = {
        "condition": cond["name"],
        "episodes": n_ep,
        "falls": len(fallen),
        "falls_per_100": 100.0 * len(fallen) / n_ep,
        "falls_per_100_ci95": wilson_ci(len(fallen), n_ep),
        "truncated": sum(e["outcome"] == "truncated" for e in episodes),
        "completed": sum(e["outcome"] == "completed" for e in episodes),
        "duration_s": stats([e["duration_s"] for e in episodes]),
        "time_to_fall_s": stats([e["fall_time_s"] for e in fallen]),
        "vel_error_m_s": stats([e["mean_vel_error_m_s"] for e in episodes]),
        "vel_error_1s_m_s": stats([e["mean_vel_error_1s_m_s"] for e in episodes
                                   if e["mean_vel_error_1s_m_s"] is not None]),
        "forward_vel_m_s": stats([e["mean_forward_vel_m_s"] for e in episodes]),
        "cmd_forward_vel_m_s": stats([e["mean_cmd_forward_vel_m_s"] for e in episodes]),
        "mean_speed_m_s": stats([e["mean_speed_m_s"] for e in episodes]),
        "abs_yaw_rate_rad_s": stats([abs(e["mean_yaw_rate_rad_s"]) for e in episodes]),
        "abs_heading_change_deg": stats([abs(e["heading_change_deg"]) for e in episodes]),
        "episodes_turning_left": sum(e["mean_yaw_rate_rad_s"] > 0.02 for e in episodes),
        "episodes_turning_right": sum(e["mean_yaw_rate_rad_s"] < -0.02 for e in episodes),
        "mean_power_W": stats([e["mean_power_W"] for e in episodes]),
        "max_tilt_deg": stats([e["max_tilt_deg"] for e in episodes]),
    }
    if summary["truncated"]:
        msg = (f"{summary['truncated']} of {n_ep} episodes ended early WITHOUT a fall (time-out or out of terrain "
               f"bounds), mean duration {summary['duration_s']['mean']:.2f} s: check the video and the episode "
               f"durations before using this condition")
        print("[WARNING]", msg)
        warnings.append(msg)

    # ---------------- saving ----------------
    local_dir = os.path.join("/content/eval_runs", os.path.basename(conf["drive_dir"].rstrip("/")), conf["run_name"])
    os.makedirs(local_dir, exist_ok=True)
    fields = {}
    if save_steps and rows:
        arr = {k: torch.cat(v).numpy() for k, v in rows.items()}
        arr["t_s"] = arr["step"].astype(np.float32) * dt
        if "foot_height" in layout_c:
            arr["foot_height"] = arr["obs_critic"][:, slice(*layout_c["foot_height"])]
        fall_step = np.full(n_ep, -1, dtype=np.int64)
        for e in fallen:
            fall_step[e["id"]] = int(round(e["duration_s"] / dt)) - 1
        fs = fall_step[arr["episode"]]
        arr["falls_within_2s"] = (fs >= 0) & (fs - arr["step"] < steps_2s)
        np.savez_compressed(os.path.join(local_dir, "steps.npz"), **arr)
        descriptions = dict(FIELD_DESCRIPTIONS)
        for k, name in enumerate(hidden_names):
            descriptions[name] = f"hidden layer {k + 1} activations after ELU, float16"
        fields = {k: {"shape": list(v.shape), "dtype": str(v.dtype), "description": descriptions.get(k, "")}
                  for k, v in arr.items()}
        print(f"[data] {len(arr['step'])} steps recorded")

    video_info = None
    if video is not None and video.frames:
        try:
            video.save(os.path.join(local_dir, "episode_video.mp4"), fps=round(1 / dt))
            first = episodes[0]
            video_info = {"file": "episode_video.mp4", "episode_id": first["id"], "outcome": first["outcome"],
                          "duration_s": first["duration_s"], "frames": len(video.frames)}
            print(f"[video] episode {first['id']} ({first['outcome']}, {first['duration_s']:.1f} s)")
        except Exception as e:  # noqa: BLE001
            msg = f"video not saved: {e}"
            print("[WARNING]", msg)
            warnings.append(msg)

    with open(conf["policy"], "rb") as f:
        md5 = hashlib.md5(f.read()).hexdigest()
    meta = {
        "run_name": conf["run_name"],
        "date": time.strftime("%Y-%m-%d %H:%M:%S"),
        "config": conf,
        "terrain_params_used": {k: list(v) if isinstance(v, tuple) else v for k, v in terrain_params.items()},
        "robot_mass_nominal_kg": scen.get("robot_mass_nominal"),
        "policy": {"name": conf.get("policy_name"), "file": os.path.basename(conf["policy"]), "md5": md5},
        "onnx_pytorch_max_diff": check_diff,
        "versions": software_versions(),
        "device": device,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "parallel_envs": num_envs,
        "control_dt_s": dt,
        "steps_per_episode": T,
        "obs_layout_actor": layout_a,
        "obs_layout_critic": layout_c,
        "video": video_info,
        "steps_file": "steps.npz" if fields else None,
        "steps_fields": fields,
        "runtime_s": time.time() - t_start,
        "warnings": sorted(set(warnings)),
    }
    with open(os.path.join(local_dir, "summary.json"), "w") as f:
        json.dump({"meta": meta, "summary": summary, "episodes": episodes}, f, indent=1)

    drive_dir = conf["drive_dir"]
    if os.path.isdir("/content/drive/MyDrive") or not drive_dir.startswith("/content/drive"):
        dest = os.path.join(drive_dir, conf["run_name"])
        shutil.copytree(local_dir, dest, dirs_exist_ok=True)
        print(f"[saved] {dest}")
    else:
        print(f"[WARNING] Drive not mounted: files kept in {local_dir}")
    print(json.dumps(summary, indent=1))
    env.close()


if __name__ == "__main__":
    if len(sys.argv) > 1 and sys.argv[1] == "--list":
        print(f"{'plane':18s} {'(training flat environment)':32s} infinite plane, default contacts and solver")
        for name, (cls, params) in CATALOG.items():
            cls_name = cls if isinstance(cls, str) else cls.__name__
            print(f"{name:18s} {cls_name:32s} {params}")
        sys.exit(0)
    with open(sys.argv[1]) as f:
        main(json.load(f))

Overwriting eval_scenario.py


In [ ]:
!uv run python eval_scenario.py --list

## 7. Define the suite

Edit the lists to choose the conditions. Every condition gets `EPISODES_PER_CONDITION` episodes. Names are built automatically, e.g. `slope_10deg_downhill`, `friction_0.3`, `payload_160g_back`.

`EXTRA_CONDITIONS` takes custom or combined conditions as dictionaries with these keys (all optional except `name`):

| Key | Meaning |
|---|---|
| `name` | folder name of the condition |
| `terrain` | a name from the list above (default `flat`); `plane` = exactly the flat environment `alpha_walking` was trained in (infinite plane, default contacts and solver), while `flat` is a generated flat terrain with the softened contacts used by all the other terrains |
| `difficulty` | 0-1, position inside the terrain parameter range (default 1.0) |
| `slope_deg` | slope angle for `terrain: "slope"` (the pyramid) |
| `direction` | with `terrain: "slope"`: `"downhill"` or `"uphill"` along the centre line of one face (sets spawn position, height and yaw) |
| `tile_size_m` | pyramid size for directional slopes (default 40) |
| `slope_start_m` | distance of the start from the pyramid centre, the same for uphill and downhill (default: a quarter of the pyramid size, i.e. the middle of the face) |
| `heading_deg` | fixed spawn yaw on other terrains; omitted = random |
| `command` | `"random"` (training distribution, default) or `{"vx": 0.3}` for a constant forward walk |
| `hold_heading` | with a constant command: `True` keeps the spawn heading with mjlab's heading controller |
| `friction` | friction of floor and feet; omitted = training randomization |
| `payload` | `{"mass_kg": 0.16, "offset_m": [x, y, z]}`, offset from the trunk centre of mass |
| `terrain_params` | overrides of the terrain parameters |

`SUITE_NAME` is fixed (no timestamp) on purpose: if Colab disconnects, re-running cell 8 skips the conditions already saved. Change it (or `SEED`) to start a new suite.

**Drive space:** with `SAVE_STEPS = True` each condition takes roughly 150-250 MB. Activations are needed for the policy you study with SAE and steering; for comparison-only runs (e.g. `rough_walk_g`) set `SAVE_STEPS = False`.

In [8]:
# ======================= SUITE =======================
EPISODES_PER_CONDITION = 100
EPISODE_LENGTH_S = 20.0
FORWARD_SPEED = 0.3                    # m/s, constant forward command for slopes and "flat_forward";
                                       # the policies reach about half of it (0.1 m/s is a dead zone)
HOLD_HEADING = True                    # keep the spawn heading on slopes and "flat_forward"

SLOPES_DEG = [0, 2.5, 5, 7.5, 10, 15]  # pyramid face; 0 = control (flat heightfield, same code)
                                       # training on rough terrain reaches ~5.7 deg
SLOPE_DIRECTIONS = ["downhill", "uphill"]
SLOPE_TILE_SIZE = 40.0                 # m: faces about 19 m long; the start is in the middle of a face,
                                       # with ~9 m available both uphill and downhill

FRICTIONS = [0.6, 0.4, 0.3, 0.2, 0.1]  # floor and feet; training feet friction is 0.7-1.3

PAYLOADS = {                           # name: (mass kg, offset from trunk CoM [x fwd, y left, z up] m)
    "payload_80g": (0.08, (0.0, 0.0, 0.03)),
    "payload_160g": (0.16, (0.0, 0.0, 0.03)),
    "payload_160g_back": (0.16, (-0.04, 0.0, 0.03)),
    "payload_160g_front": (0.16, (0.04, 0.0, 0.03)),
    "payload_160g_side": (0.16, (0.0, 0.04, 0.03)),
}

TERRAINS = {                           # name: difficulty (0-1)
    "slope": 1.0,                      # pyramid ~10 deg, random walking directions (8 m patches)
    "pyramid_stairs": 1.0,             # steps up to 1.5 cm
    "random_stairs": 1.0,
    "open_stairs": 1.0,
    "random_grid": 1.0,                # cobblestones up to 1 cm
    "random_uniform": 1.0,             # bumps up to 2 cm
    "waves": 1.0,
    "perlin": 1.0,
    "tilted_grid": 1.0,
    "stepping_stones": 1.0,            # these last five use mjlab defaults or guessed parameters:
    "narrow_beams": 1.0,               # check "terrain_params_used" and "warnings" in summary.json
    "nested_rings": 1.0,
    "random_spread": 1.0,
    "discrete_obstacles": 1.0,
}

EXTRA_CONDITIONS = [
    # {"name": "slope_10deg_downhill_friction_0.3", "terrain": "slope", "slope_deg": 10,
    #  "direction": "downhill", "command": {"vx": 0.3}, "hold_heading": True, "friction": 0.3},
]
ONLY = [f"slope_{deg}deg_{direction}" for deg in SLOPES_DEG for direction in SLOPE_DIRECTIONS]                    # e.g. ["flat", "slope_10deg_downhill"] to run a subset
PUSHES = False
SAVE_STEPS = True
SAVE_VIDEO = True                      # video of one evaluated episode per condition (a few MB each)
FLOOR_TEXTURE = True                   # 20 cm checkerboard on the terrain, visual only (no physical effect)
SEED = 0
MAX_ENVS = 256                         # copies simulated in parallel; 256 is safe on a T4
DRIVE_DIR = "/content/drive/MyDrive/microduck_thesis/eval"
SUITE_NAME = f"{POLICY}_seed{SEED}_piramide_v2"   # v2: pyramid slopes and heading hold (older suites used the ramp)
# =====================================================

CONDITIONS = [
    {"name": "flat"},
    {"name": "flat_forward", "heading_deg": 0, "command": {"vx": FORWARD_SPEED}, "hold_heading": HOLD_HEADING},
]
for deg in SLOPES_DEG:
    for direction in SLOPE_DIRECTIONS:
        CONDITIONS.append({"name": f"slope_{deg}deg_{direction}", "terrain": "slope", "slope_deg": deg,
                           "direction": direction, "tile_size_m": SLOPE_TILE_SIZE,
                           "command": {"vx": FORWARD_SPEED}, "hold_heading": HOLD_HEADING})
for mu in FRICTIONS:
    CONDITIONS.append({"name": f"friction_{mu}", "friction": mu})
for name, (mass, offset) in PAYLOADS.items():
    CONDITIONS.append({"name": name, "payload": {"mass_kg": mass, "offset_m": list(offset)}})
for terrain, difficulty in TERRAINS.items():
    CONDITIONS.append({"name": f"{terrain}_d{difficulty}", "terrain": terrain, "difficulty": difficulty})
CONDITIONS += EXTRA_CONDITIONS

names = [c["name"] for c in CONDITIONS]
assert len(names) == len(set(names)), "condition names must be unique"
selected = [c for c in CONDITIONS if not ONLY or c["name"] in ONLY]
print(f"suite '{SUITE_NAME}': {len(selected)} conditions x {EPISODES_PER_CONDITION} episodes "
      f"= {len(selected) * EPISODES_PER_CONDITION} episodes")
for c in selected:
    print(" -", c["name"])

suite 'alpha_walking_seed0_piramide_v2': 12 conditions x 100 episodes = 1200 episodes
 - slope_0deg_downhill
 - slope_0deg_uphill
 - slope_2.5deg_downhill
 - slope_2.5deg_uphill
 - slope_5deg_downhill
 - slope_5deg_uphill
 - slope_7.5deg_downhill
 - slope_7.5deg_uphill
 - slope_10deg_downhill
 - slope_10deg_uphill
 - slope_15deg_downhill
 - slope_15deg_uphill


## 8. Run the suite

Runs every selected condition in sequence and then shows the results table. The first condition also compiles the MuJoCo Warp kernels (a few minutes); the following ones reuse them. Conditions already saved in the suite folder are skipped: **after changing the script, delete the old condition folders** (or change `SUITE_NAME`), otherwise the old results are kept.

How to read the table:
- `truncated` must be **0** and `mean_duration_s` close to `EPISODE_LENGTH_S` (minus the falls): episodes that end early without a fall mean the condition was not simulated correctly, and its numbers must not be used;
- `max_tilt_deg` is the mean over episodes of the largest trunk tilt from vertical: it shows how close the robot came to falling, also when it did not fall;
- velocity, all in the robot frame:
  - `vel_error_m_s` is the **instantaneous** error |velocity − command|. The trunk of a small biped swings forward/back and sideways within every step, so this error stays around 0.2 m/s even when the average speed is tracked well: it mostly measures the gait oscillation, not the tracking;
  - `vel_error_1s_m_s` compares velocity and command both **averaged over about 1 s** (after a 1 s warm-up): this is the tracking error to report;
  - `cmd_fwd_m_s` and `fwd_vel_m_s` are the mean commanded and actual forward velocity: with a constant command (`flat_forward`, slopes) they should be close;
  - `speed_m_s` is the distance actually covered divided by the episode duration;
- heading:
  - `yaw_rate_abs` is the mean over episodes of |mean yaw rate| (rad/s), `heading_change_deg` the mean |heading change| over the episode, `left/right` how many episodes turn left / right (|mean yaw rate| > 0.02 rad/s);
  - with heading hold (slopes, `flat_forward`) they show how well the heading is kept; with random commands they include the commanded turns.

In [9]:
import json
import pandas as pd

suite_dir = os.path.join(DRIVE_DIR, SUITE_NAME)
os.makedirs(suite_dir, exist_ok=True)

for i, cond in enumerate(selected, 1):
    if os.path.exists(os.path.join(suite_dir, cond["name"], "summary.json")):
        print(f"[{i}/{len(selected)}] {cond['name']}: already done, skipped")
        continue
    conf = {
        "run_name": cond["name"], "policy": POLICY_PATH, "policy_name": POLICY, "condition": cond,
        "num_episodes": EPISODES_PER_CONDITION, "episode_length_s": EPISODE_LENGTH_S,
        "pushes": PUSHES, "save_steps": SAVE_STEPS, "save_video": SAVE_VIDEO, "floor_texture": FLOOR_TEXTURE,
        "seed": SEED, "max_envs": MAX_ENVS, "drive_dir": suite_dir,
    }
    with open("eval_config.json", "w") as f:
        json.dump(conf, f, indent=1)
    print(f"\n[{i}/{len(selected)}] {cond['name']}")
    !uv run --with "imageio[ffmpeg]" python eval_scenario.py eval_config.json

# ---- results table ----
results, missing, warnings = [], [], {}
for cond in CONDITIONS:
    path = os.path.join(suite_dir, cond["name"], "summary.json")
    if not os.path.exists(path):
        missing.append(cond["name"])
        continue
    with open(path) as f:
        run = json.load(f)
    s = run["summary"]
    mean = lambda key: s[key]["mean"] if s.get(key) else None   # noqa: E731  (older runs lack some keys)
    low, high = s["falls_per_100_ci95"]
    results.append({
        "condition": cond["name"], "episodes": s["episodes"], "falls_per_100": s["falls_per_100"],
        "ci95_low": low, "ci95_high": high,
        "truncated": s["truncated"],              # must be 0: otherwise the condition is not valid
        "mean_duration_s": mean("duration_s"),
        "max_tilt_deg": mean("max_tilt_deg"),
        "vel_error_m_s": mean("vel_error_m_s"),   # instantaneous, includes the gait oscillation
        "vel_error_1s_m_s": mean("vel_error_1s_m_s"),
        "cmd_fwd_m_s": mean("cmd_forward_vel_m_s"),
        "fwd_vel_m_s": mean("forward_vel_m_s"),
        "speed_m_s": mean("mean_speed_m_s"),
        "yaw_rate_abs": mean("abs_yaw_rate_rad_s"),
        "heading_change_deg": mean("abs_heading_change_deg"),
        "left/right": f"{s.get('episodes_turning_left', '-')}/{s.get('episodes_turning_right', '-')}",
        "power_W": mean("mean_power_W"),
    })
    if run["meta"]["warnings"]:
        warnings[cond["name"]] = run["meta"]["warnings"]

with open(os.path.join(suite_dir, "suite_summary.json"), "w") as f:
    json.dump({"policy": POLICY, "policy_path": POLICY_PATH, "seed": SEED,
               "episodes_per_condition": EPISODES_PER_CONDITION, "episode_length_s": EPISODE_LENGTH_S,
               "conditions": CONDITIONS, "results": results}, f, indent=1)

display(pd.DataFrame(results).round(3))
if missing:
    print("not available (failed or not run):", ", ".join(missing))
for name, w in warnings.items():
    print(f"WARNINGS in {name}:", *w, sep="\n - ")


[1/12] slope_0deg_downhill
[mdp] Patches 1-2 active: NaN-safe reward/advantage
[mdp] Patch 4 active: ONNX export filters passive_* joints
[mdp] Patch 5 active: MICRODUCK_WARM_START=1 restarts curricula after checkpoint load
=== condition 'slope_0deg_downhill' | policy alpha_walking.onnx | 100 episodes ===
[slope] 0 deg downhill: spawn 10.00 m from the pyramid centre, -0.001 m from the top platform height, yaw 0 deg
Setting seed: 0
Warp 1.12.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.0
Terrain generation took 0.0163 seconds.
/content/microduck_rl/.venv/lib/python3.12/site-packages/mjlab/entity/entity.py:149: UserWarning: Actuator config matched 14 joint(s) but the same expressions also match 7 site(s): imu, left_foot, head_camera, tof, head_imu, mouth_tip, right_foot. Add a separate config with transmission_type=TransmissionType.SITE if

,condition,episodes,falls_per_100,ci95_low,ci95_high,truncated,mean_duration_s,max_tilt_deg,vel_error_m_s,vel_error_1s_m_s,cmd_fwd_m_s,fwd_vel_m_s,speed_m_s,yaw_rate_abs,heading_change_deg,left/right,power_W
0,slope_0deg_downhill,100,0.0,0.000,3.699,0,20.000,6.089,0.212,0.152,0.3,0.151,0.149,0.006,7.436,3/1,1.076
1,slope_0deg_uphill,100,0.0,0.000,3.699,0,20.000,6.050,0.212,0.152,0.3,0.151,0.149,0.007,7.896,3/2,1.078
2,slope_2.5deg_downhill,100,0.0,0.000,3.699,0,20.000,6.807,0.174,0.103,0.3,0.202,0.200,0.009,9.934,10/0,1.232
3,slope_2.5deg_uphill,100,0.0,0.000,3.699,0,20.000,6.065,0.246,0.194,0.3,0.108,0.107,0.008,9.351,7/2,0.995
4,slope_5deg_downhill,100,0.0,0.000,3.699,0,20.000,8.435,0.146,0.070,0.3,0.248,0.248,0.013,14.591,20/1,1.539
5,slope_5deg_uphill,100,0.0,0.000,3.699,0,20.000,7.521,0.283,0.239,0.3,0.063,0.063,0.012,13.578,15/0,0.926
6,slope_7.5deg_downhill,100,4.0,1.566,9.837,0,19.683,12.397,0.138,0.062,0.3,0.288,0.290,0.016,13.996,12/3,2.360
7,slope_7.5deg_uphill,100,13.0,7.757,20.981,0,19.228,18.642,0.316,0.276,0.3,0.029,0.047,0.030,28.010,34/7,1.226
8,slope_10deg_downhill,100,26.0,18.405,35.371,0,16.785,26.320,0.173,0.079,0.3,0.353,0.363,0.078,26.167,13/21,4.374
9,slope_10deg_uphill,100,67.0,57.305,75.437,0,13.146,49.325,0.387,0.343,0.3,-0.035,0.098,0.118,57.423,28/42,2.029


not available (failed or not run): flat, flat_forward, friction_0.6, friction_0.4, friction_0.3, friction_0.2, friction_0.1, payload_80g, payload_160g, payload_160g_back, payload_160g_front, payload_160g_side, slope_d1.0, pyramid_stairs_d1.0, random_stairs_d1.0, open_stairs_d1.0, random_grid_d1.0, random_uniform_d1.0, waves_d1.0, perlin_d1.0, tilted_grid_d1.0, stepping_stones_d1.0, narrow_beams_d1.0, nested_rings_d1.0, random_spread_d1.0, discrete_obstacles_d1.0


## 9. Compare two policies

Run the suite once per policy (change `POLICY` in cell 5, re-run cells 5, 7 and 8), then compare the two suites. `A worse` marks the conditions where the 95% intervals of the fall rate do not overlap and A falls more: a conservative test, so a missing mark does not mean "no difference".

In [ ]:
SUITE_A = "alpha_walking_seed0_v2"
SUITE_B = "rough_walk_g_seed0_v2"

def load_suite(name):
    with open(os.path.join(DRIVE_DIR, name, "suite_summary.json")) as f:
        return pd.DataFrame(json.load(f)["results"]).set_index("condition")

a, b = load_suite(SUITE_A), load_suite(SUITE_B)
cols = [c for c in ["falls_per_100", "ci95_low", "ci95_high", "vel_error_1s_m_s", "cmd_fwd_m_s",
                     "fwd_vel_m_s", "yaw_rate_abs", "power_W"] if c in a.columns and c in b.columns]
comparison = a[cols].join(b[cols], lsuffix=" A", rsuffix=" B", how="outer")
comparison["falls diff (A-B)"] = comparison["falls_per_100 A"] - comparison["falls_per_100 B"]
comparison["A worse"] = comparison["ci95_low A"] > comparison["ci95_high B"]
print(f"A = {SUITE_A}   B = {SUITE_B}")
display(comparison.round(2))

## 10. Reading the data back

Example: one condition's episodes in a table and its steps as NumPy arrays. Works in a fresh session too: mount Drive and set the suite and condition names.

In [ ]:
import numpy as np

run_dir = os.path.join(DRIVE_DIR, SUITE_NAME, "flat")   # any condition folder

with open(os.path.join(run_dir, "summary.json")) as f:
    run = json.load(f)
episodes = pd.DataFrame(run["episodes"])
display(episodes.head())

if os.path.exists(os.path.join(run_dir, "steps.npz")):
    steps = np.load(os.path.join(run_dir, "steps.npz"))
    for name in steps.files:
        print(f"{name:18s} {str(steps[name].shape):16s} {steps[name].dtype}")
    # e.g. second hidden layer activations in the steps that precede a fall
    h2_before_fall = steps["h2"][steps["falls_within_2s"]]
    print("steps less than 2 s before a fall:", len(h2_before_fall))

### Velocity check

Uses `steps.npz` (also from runs made before the new velocity metrics) to see whether the robot walks at the commanded speed. With a constant command (`flat_forward`, slopes):
- **mean actual vx close to the command** and a large standard deviation → the robot tracks the speed and the instantaneous error is the gait oscillation;
- **mean actual vx close to 0** → the robot steps in place or barely moves: the policy does not follow the command (or the command is not the one we think).

In [ ]:
CHECK_CONDITION = "flat_forward"

run_dir = os.path.join(DRIVE_DIR, SUITE_NAME, CHECK_CONDITION)
steps = np.load(os.path.join(run_dir, "steps.npz"))
with open(os.path.join(run_dir, "summary.json")) as f:
    layout = json.load(f)["meta"]["obs_layout_actor"]
cmd = steps["obs_actor"][:, slice(*layout.get("command", [48, 51]))]
vel = steps["lin_vel_b"]
settled = steps["t_s"] >= 1.0   # skip the first second after the spawn

print(f"commanded vx: mean {cmd[settled, 0].mean():+.3f} m/s")
print(f"actual    vx: mean {vel[settled, 0].mean():+.3f} m/s   std {vel[settled, 0].std():.3f}")
print(f"actual    vy: mean {vel[settled, 1].mean():+.3f} m/s   std {vel[settled, 1].std():.3f}")
inst = np.linalg.norm(vel[settled, :2] - cmd[settled, :2], axis=1).mean()
print(f"instantaneous error |v - cmd|: {inst:.3f} m/s")

# per-episode mean velocity: averaging over the episode removes the within-step oscillation
episode = steps["episode"][settled]
count = np.bincount(episode)
ok = count > 0
mean_vx = np.bincount(episode, vel[settled, 0]) / np.maximum(count, 1)
mean_cmd = np.bincount(episode, cmd[settled, 0]) / np.maximum(count, 1)
print(f"per-episode mean vx: {mean_vx[ok].mean():+.3f} m/s (min {mean_vx[ok].min():+.3f}, "
      f"max {mean_vx[ok].max():+.3f}) vs command {mean_cmd[ok].mean():+.3f} m/s")

## 11. Watching the videos

Shows the saved episode video of every condition in the suite (or only those in `SHOW`), with its outcome.

In [ ]:
from base64 import b64encode
from IPython.display import HTML

SHOW = None   # e.g. ["flat", "slope_10deg_downhill"]; None = every condition with a video

for cond in CONDITIONS:
    if SHOW and cond["name"] not in SHOW:
        continue
    run_dir = os.path.join(DRIVE_DIR, SUITE_NAME, cond["name"])
    video_path = os.path.join(run_dir, "episode_video.mp4")
    if not os.path.exists(video_path):
        continue
    with open(os.path.join(run_dir, "summary.json")) as f:
        info = json.load(f)["meta"].get("video") or {}
    with open(video_path, "rb") as f:
        data = b64encode(f.read()).decode()
    display(HTML(f'<p><b>{cond["name"]}</b>: episode {info.get("episode_id")}, {info.get("outcome")}, '
                 f'{info.get("duration_s", 0):.1f} s</p>'
                 f'<video width=480 controls src="data:video/mp4;base64,{data}"></video>'))

## Troubleshooting

| Message | Likely cause | What to do |
|---|---|---|
| `[WARNING] ... using mjlab defaults` | The Microduck-scaled parameter names do not exist for that terrain type | The message lists the available fields: pass them in `terrain_params` |
| `The PyTorch policy does not match the ONNX one` | Wrong ONNX → PyTorch conversion | Do not use the data; paste the error |
| `unsupported ONNX op` | The graph contains an unexpected operation | Paste the error: it gets added to `load_policy` |
| `expected 2 foot collision geoms` | Foot geoms not found in the scene | Paste the error with the printed names |
| Uphill slopes: falls in the first second (`time_to_fall_s` ~0-1 s), or the video shows the robot dropping onto / sinking into the face | Spawn height on the pyramid face not matching the surface | Check the `[slope]` line printed by the script and `terrain_params_used.spawn` in `summary.json`; paste them |
| `AttributeError` on `heading_command` or `ranges.heading` | Different heading-controller fields in this mjlab version | Paste the error; meanwhile set `HOLD_HEADING = False` |
| `trunk_base body not found` | The trunk has a different name in the scene | Paste the error |
| `foot_friction` / `terrain_friction` in `summary.json` differ from the condition | The friction was overwritten after the scene was built | Do not use that condition; paste the values |
| `AttributeError` on `rel_turn_in_place_envs` or `ranges` | Different field names in the command | Paste the error |
| `WARNINGS: ... not available` | A scenario field was not read | The walking data are still valid; that field is empty in the JSON |
| `video disabled, the renderer could not start` | EGL not available in the evaluation process | Re-run cell 3 before cell 8; the evaluation data are still valid |
| No `episode_video.mp4` for some conditions | Those conditions were run before the video was added, and cell 8 skips them | Delete their folders on Drive (or change `SUITE_NAME`) and re-run cell 8 |
| CUDA out of memory | Too many parallel copies | Lower `MAX_ENVS` |
| Many falls even on `flat` | Policy and observations not aligned | Check that you use a 61-observation walking policy |